# Bias-Varianz-Abwägung



Die Bias-Varianz-Abwägung ist ein zentrales Problem im überwachten Lernen. Der Fehler eines maschinellen Lernmodells besteht aus 3 Teilen:

$MSE = Varianz + Bias^2 + Rauschen$ 

Das Rauschen wird auch als *nicht reduzierbarer Fehler* bezeichnet, es kann nicht durch die Anwendung verschiedener Modelle reduziert werden.
Der Varianz- und Bias-Term wird durch die Flexibilität des gewählten Modells beeinflusst.

In [ ]:
from IPython.display import HTML
url = 'https://upload.wikimedia.org/wikipedia/commons/9/9f/Bias_and_variance_contributing_to_total_error.svg'
HTML(f'<img src="{url}" width="600">')

Normalerweise ist unser Ziel, den Gesamtfehler (z.B. den mittleren quadratischen Fehler) zu minimieren. Daher möchten wir den Bias- und Varianz-Term des Fehlers minimieren.

Leider ist es in der Regel unmöglich, beides gleichzeitig zu tun. **Lernmethoden mit hoher Varianz** können ihren Trainingssatz gut repräsentieren, laufen aber Gefahr, auf verrauschte oder nicht repräsentative Trainingsdaten **überzupassen** (Overfitting). Im Gegensatz dazu erzeugen Algorithmen mit **hohem Bias** typischerweise einfachere Modelle, die möglicherweise wichtige Regelmäßigkeiten in den Daten nicht erfassen (d.h. **Underfitting**).

Dies wird als **Bias-Varianz-Abwägung** bezeichnet. Was es bedeutet, ist, dass es sinnvoll sein kann, einen verzerrten Schätzer zu verwenden, solange er unsere Varianz reduziert, vorausgesetzt, unser Ziel ist es, den quadratischen Fehler zu minimieren.

Lernziele

In diesem Notebook lernen Sie...

* wie Sie Ihre Daten mit `train_test_split` in einen Trainings- und Testsatz aufteilen
* wie Sie flexiblere lineare Modelle erstellen, indem Sie Polynommerkmale verwenden
* wie Sie den mittleren quadratischen Fehler (RMSE) zur Bewertung der Leistung Ihres Modells verwenden
* Bias und Varianz zu definieren
* was die Bias-Varianz-Abwägung ist und ihre Auswirkungen an einem konkreten Beispiel zu sehen

## Setup und Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import operator
# import to divide our data into train and test data
from sklearn.model_selection import train_test_split
# import to create polynomial features
from sklearn.preprocessing import PolynomialFeatures
# import of the linear regression model
from sklearn.linear_model import LinearRegression
# import of our evaluation metrics
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    root_mean_squared_error,  # available in scikit-learn >= 1.4
)
RSEED = 12

In [ ]:
# Set constants and plotting parameters
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['figure.titlesize'] = 18
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.figsize'] = (8,6)

## Definieren unserer Daten




Normalerweise würden wir die wahre Beziehung zwischen den unabhängigen Merkmalen (X) und unserer abhängigen Variable (y) nicht kennen. Aber um eine Vorstellung von der Bias-Varianz-Abwägung zu bekommen, ist es hilfreich, die wahre Beziehung zwischen diesen Variablen zu kennen.
Hier kennen wir diese Beziehung, weil wir sie in unserem Datengenerierungsprozess definieren.

Wir wählen x im Bereich von -7 bis 7. Und die wahre Beziehung zwischen x und y ist:


$y = 10x - 2 x^2 + 0.5 x^3 + \epsilon$

Das $\epsilon$ führt etwas zufälliges und unvorhersehbares Rauschen in unsere Daten ein.



In [ ]:
# generate random data 
np.random.seed(4)
# generating 100 random x-values between -7 and 7
x = np.random.uniform(-7,7,size=100)

# generating corresponding y variable according to our defined true relationship
# adding some random noise with np.random.normal with mu = 0 and std = 20
y = 10 * x - 2 * (x ** 2) + 0.5 * (x ** 3) + np.random.normal(0, 20, 100)

# transforming the data to include another axis since this is expected for modelling with scikit-learn
x = x[:, np.newaxis]
y = y[:, np.newaxis]

## Train-Test-Aufteilung



Meistens ist ein Ziel Ihrer Data-Science-Projekte, ein Modell zu trainieren, um Vorhersagen für neue Instanzen zu treffen. Aber das Training eines Modells allein garantiert nicht, dass es auf unbekannten Daten gut funktioniert. Bevor wir unserem Modell "vertrauen" und es in Produktion bringen können, müssen wir seine Leistung auf Daten bewerten, auf denen es **nicht** trainiert wurde. 

Der einfachste Weg, dies zu tun, ist, unseren ursprünglichen Datensatz in einen Trainingsdatensatz und einen Testdatensatz aufzuteilen. Der Trainingsdatensatz wird, wie der Name schon sagt, verwendet, um unser Modell anzupassen (zu trainieren). Wir können dann dieses Modell gegen den Testdatensatz bewerten, der nur Instanzen enthält, die unser Modell noch nie gesehen hat. Glücklicherweise müssen wir unseren Datensatz nicht manuell aufteilen. Wir können die [`train-test-split()`-Funktion](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) von scikit-learn verwenden.

In [ ]:
# Split data into train and test set
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.8, random_state=RSEED) #Careful here: this is an unusual choice for the test_size. See paragraph below for more explainations.


In [ ]:
# Size of train and test set
print("X_train: ", X_train.shape)
print("y_train: ", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

In den obigen Zellen haben wir die `train_test_split()`-Funktion auf unsere Merkmale und Zielvariable aufgerufen. Wir haben auch definiert, welcher Prozentsatz des gesamten Datensatzes im Testsatz landen soll (`test_size`).


Ein weiterer wichtiger Parameter ist `random_state`: Ohne Festlegung eines Random State wird scikit-learn Ihre Daten beim Aufruf der Funktion **zufällig** mischen und Instanzen dem Testsatz zuweisen. Dadurch ändern sich die Scores Ihres Modells bei mehreren Funktionsaufrufen. Um bei jedem Aufruf eines Notebooks oder Skripts dieselben Ergebnisse zu reproduzieren, müssen Sie das Mischen kontrollieren, das auf die Daten angewendet wird. Sie können dies tun, indem Sie den Parameter `random_state` setzen. 


    Übung:

    Die Dokumentation von scikit-learn ist sehr wertvoll. Werfen Sie einen Blick auf die scikit-learn-Seite für train-test-split. Was ist der Standardwert für test_size? 

Die Standardwerte von scikit-learn sind immer ein guter Ausgangspunkt und eine Benchmark. Warum weichen wir hier also so stark von der Norm ab? 

Es gibt zwei konkurrierende Bedenken: Mit weniger Trainingsdaten haben Ihre Parameterschätzungen eine größere Varianz. Mit weniger Testdaten hat Ihre Leistungsstatistik eine größere Varianz. Grob gesagt sollten Sie sich darum kümmern, die Daten so aufzuteilen, dass keine der beiden Varianzen zu hoch ist, was mehr mit der absoluten Anzahl von Instanzen in jeder Kategorie zu tun hat als mit dem Prozentsatz.

In unserem Fall hier wollen wir zeigen, was "größere Varianz" für den geschätzten Parameter bedeutet, daher ist unsere Aufteilung etwas unkonventionell. 


Schauen wir uns unsere Daten an. Es ist immer gut, ein Gefühl für die Trainings- und Testdaten zu bekommen und zu prüfen, ob diese aus derselben Verteilung stammen oder stark variieren.

In [ ]:
x_plot = np.linspace(-7,7,500)
y_plot_true = 10 * x_plot - 2 * (x_plot ** 2) + 0.5 * (x_plot ** 3)

In [ ]:
# Plotting 
plt.scatter(X_train, y_train, s=10,c="#193251",alpha=0.7, label="Train data")
plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",label= '"True" Relationship')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Generated Data")
plt.show();

Die wahre Beziehung ist genau unser Datengenerierungsprozess:


$y = 10x - 2 x^2 + 0.5 x^3 + \epsilon$


Die Daten weichen von dieser Linie durch den zufälligen Fehler ab, den wir ebenfalls hinzugefügt haben.

Insgesamt erstrecken sich die Trainingsdaten über nahezu denselben Bereich im Merkmal x wie die Testdaten. Nur im sehr negativen Bereich (< -6) gibt es keine Trainingsdaten.

## Modellierung


Wir werden verschiedene Modelle auf unseren generierten Daten ausprobieren. Durch die Verwendung unterschiedlich flexibler Modelle werden wir auch die Auswirkungen von Bias und Varianz dieser Modelle sehen.



### Erstes Modell: Lineare Regression


Nachdem wir unsere Merkmale und Zielvariable definiert und die Daten in Trainings- und Testsätze aufgeteilt haben, können wir mit der Modellierung beginnen. Als ersten Schritt erstellen wir eine Instanz von `LinearRegression()` und passen sie mit `.fit()` an die **Trainingsdaten** an. Dann machen wir Vorhersagen für **Trainings-** und **Testsätze** mit `.predict()`.

> Hinweis: scikit-learn bietet `root_mean_squared_error` (seit v1.4) zur direkten Berechnung des RMSE. Dies ist der empfohlene Weg in neueren Versionen (z.B. v1.7.2) anstelle der Verwendung von `mean_squared_error(..., squared=False)`.

Wir werden **R²** und **RMSE** sowohl für Training als auch Test berichten, um die Modellleistung zu bewerten.

In [ ]:
# Instantiate our model
lin_reg = LinearRegression()

# Fit the model using our train data
lin_reg.fit(X_train, y_train)

# Make predictions on both train and test
y_pred_train = lin_reg.predict(X_train)
y_pred_test  = lin_reg.predict(X_test)


Bisher haben wir ein Modell trainiert und Vorhersagen für unseren Testsatz gemacht. Jetzt ist es an der Zeit, die Leistung des Modells zu bewerten. Wir werden das R-Quadrat und den RMSE ausgeben.
Wir werden `root_mean_squared_error` von `sklearn.metrics` verwenden, um den RMSE zu berechnen.
Falls Sie `mean_squared_error` verwenden, können Sie den resultierenden Wert durch eine Quadratwurzelfunktion (d.h. `np.sqrt()`) leiten, um den RMSE zu erhalten.

In [ ]:
# Calculate metric
def calculate_metrics(y_train, y_pred_train, y_test, y_pred_test):
    """Calculate and print out RMSE and R2 for train and test data

    Args:
        y_train (array): true values of y_train
        y_pred_train (array): predicted values of model for y_train
        y_test (array): true values of y_test
        y_pred_test (array): predicted values of model for y_test
    """

    print("Metrics on training data")
    rmse_train = root_mean_squared_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)
    print("RMSE:", round(rmse_train, 3))
    print("R2:", round(r2_train, 3))
    print("---" * 10)

    print("Metrics on test data")
    rmse_test = root_mean_squared_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)
    print("RMSE:", round(rmse_test, 3))
    print("R2:", round(r2_test, 3))
    print("---" * 10)

In [ ]:
print("Linear Regression Model:")
print("---" * 10)
calculate_metrics(y_train, y_pred_train, y_test, y_pred_test)

print('Standard deviation of y:', np.std(y))
print('Mean of y:', np.mean(y))


Der optimale RMSE wäre 0, was bedeutet, dass alle Vorhersagen genau mit den erwarteten Werten übereinstimmen.
Hier sehen wir, dass der RMSE auf den Trainingsdaten mit 33 kleiner ist als der RMSE auf den Testdaten (44).
Dies ist typisch, da wir unsere Koeffizienten basierend auf den Trainingsdaten berechnet haben, ist das Modell anfällig dafür, die Trainingsdaten besser anzupassen als die unbekannten Testdaten.

Schauen wir uns unser Modell grafisch an.

In [ ]:
line_lin_reg = lin_reg.intercept_ + lin_reg.coef_[0]*x_plot

In [ ]:
# Plotting 
plt.scatter(X_train, y_train, s=10,c="#193251",alpha=0.7, label="Train data")
plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",label= '"True" Relationship')
plt.plot(x_plot,line_lin_reg,c="#FFBA08",label= 'Linear model')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Model Comparison")
plt.show();

Wir können sehen, dass die gerade Linie nicht in der Lage ist, die Muster in den Daten zu erfassen. Dies ist ein Beispiel für **Underfitting**. Das lineare Modell ist nicht flexibel genug, um die wahre Beziehung von x und y zu erfassen. Dies wird als **verzerrtes** Modell bezeichnet. Das Modell hat starke Vorannahmen über die Beziehung zwischen x und y: dass es eine lineare Beziehung gibt.

Mit nur einem Merkmal können wir sehen, wie unser Modell funktioniert, indem wir uns das Merkmal-Ziel-Diagramm ansehen. Aber wenn wir mehrere Merkmale haben, ist diese Grafik für uns nicht mehr verfügbar. Um dennoch die Modellleistung bewerten zu können, können wir die wahren y-Werte gegen die vorhergesagten Werte plotten und unsere Residuen plotten. Mit diesen Diagrammen überprüfen wir, ob die Residuen zufällig normalverteilt sind (ideale Situation) oder ob wir ein Muster sehen (noch Verbesserung des Modells möglich!).

In [ ]:

def error_analysis(y_test, y_pred_test):
    """Generated true vs. predicted values and residual scatter plot for models

    Args:
        y_test (array): true values for y_test
        y_pred_test (array): predicted values of model for y_test
    """     
    # Calculate residuals
    residuals = y_test - y_pred_test
    
    # Plot real vs. predicted values 
    fig, ax = plt.subplots(1,2, figsize=(15, 5))
    plt.subplots_adjust(right=1)
    plt.suptitle('Error Analysis')
    
    ax[0].scatter(y_pred_test, y_test, color="#FF5A36", alpha=0.7)
    ax[0].plot([-400, 350], [-400, 350], color="#193251")
    ax[0].set_title("True vs. predicted values", fontsize=16)
    ax[0].set_xlabel("predicted values")
    ax[0].set_ylabel("true values")
    ax[0].set_xlim((y_pred_test.min()-10), (y_pred_test.max()+10))
    ax[0].set_ylim((y_test.min()-40), (y_test.max()+40))
    
    ax[1].scatter(y_pred_test, residuals, color="#FF5A36", alpha=0.7)
    ax[1].plot([-400, 350], [0,0], color="#193251")
    ax[1].set_title("Residual Scatter Plot", fontsize=16)
    ax[1].set_xlabel("predicted values")
    ax[1].set_ylabel("residuals")
    ax[1].set_xlim((y_pred_test.min()-10), (y_pred_test.max()+10))
    ax[1].set_ylim((residuals.min()-10), (residuals.max()+10));

In [ ]:
error_analysis(y_test, y_pred_test)

Da

Residuum = Beobachtet – Vorhergesagt

...bedeuten positive Werte für das Residuum (auf der y-Achse), dass die Vorhersage zu niedrig war, und negative Werte bedeuten, dass die Vorhersage zu hoch war; 0 bedeutet, dass die Vorhersage genau richtig war.

Optimalerweise sind die Residuen,
- (1) ziemlich symmetrisch verteilt und tendieren dazu, sich zur Mitte des Diagramms zu gruppieren.
- (2) sie sind um die niedrigen einstelligen Werte der y-Achse gruppiert (z.B. 0,5 oder 1,5, nicht 30 oder 150).
- (3) im Allgemeinen gibt es keine klaren Muster.

Ok. Wir sehen immer noch einige Muster, die unser Modell nicht erfasst. Für x-Werte unter -150 überschätzen wir immer den y-Wert, zwischen -150 und circa 0 unterschätzen wir immer den y-Wert. Für x-Werte über 0 wird der Zielwert wieder allgemein überschätzt.

Dies zeigt erneut das Underfitting unseres hochgradig verzerrten Modells.



### Zweites Modell: Lineare Regression mit Polynommerkmalen


Um Underfitting zu überwinden, müssen wir die Komplexität des Modells erhöhen. Wir können dies tun, indem wir eine Gleichung höherer Ordnung durch Hinzufügen von Potenzen der ursprünglichen Merkmale als neue Merkmale erzeugen. 
Das lineare Modell, 


$\hat{y} = b_0 + b_1x $


kann transformiert werden zu

$\hat{y} = b_0 + b_1x + b_2x^2$


Dies wird immer noch als lineares Modell betrachtet, da die **Koeffizienten/Gewichte**, die mit den Merkmalen verbunden sind, immer noch linear sind. 

**x²** ist nur ein Merkmal. 

Allerdings ist die Kurve, die wir anpassen, quadratischer Natur. Um die ursprünglichen Merkmale in ihre höherwertigen Terme umzuwandeln, verwenden wir die Klasse `PolynomialFeature` von scikit-learn.

In [ ]:
# here we will add the second degree polynomial features
polynomial_features = PolynomialFeatures(degree=2, include_bias=False)
x_poly_train = polynomial_features.fit_transform(X_train)
x_poly_test = polynomial_features.transform(X_test)

model_poly = LinearRegression()
model_poly.fit(x_poly_train, y_train)
y_pred_test = model_poly.predict(x_poly_test)
y_pred_train = model_poly.predict(x_poly_train)

In [ ]:
print("Polynomial Regression Model (degree 2):")
print("---"*10)
calculate_metrics(y_train,y_pred_train, y_test, y_pred_test)

Wir können sehen, dass der RMSE der Trainings- sowie der Testdaten durch Hinzufügen quadratischer Merkmale reduziert werden konnte.

In [ ]:
line_poly_2_reg = model_poly.intercept_ + model_poly.coef_[0][0] * x_plot + model_poly.coef_[0][1] * x_plot**2

In [ ]:
# Plotting 
plt.scatter(X_train, y_train, s=10,c="#193251",alpha=0.7, label="Train data")
plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",alpha=0.7, label= '"True" Relationship')
plt.plot(x_plot,line_lin_reg,c="#FFBA08",alpha=0.5,label= 'Linear model')
plt.plot(x_plot,line_poly_2_reg,c="#FFBA08",marker = "o",ms=1, alpha=0.8,label= 'Degree 2 model')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Model Comparison")
plt.show();

Das Anpassen eines linearen Regressionsmodells auf die transformierten Merkmale ergibt das obige Diagramm. Es ist ziemlich klar aus dem Diagramm, dass die quadratische Kurve die Daten besser anpassen kann als die lineare Linie. Wenn wir den `RMSE` und den `R²-Score` der beiden Modelle vergleichen, können wir auch die Verbesserung sehen:

| Metrik   | RMSE  | R²  | 
|---|---|---|
| **Lineares Modell**   | 44       |           0.79 | 
| **Polynomiales Modell (Grad 2)**     |34     |            0.87 | 





> Wir können sehen, dass der RMSE abgenommen und der R²-Score im Vergleich zur linearen Linie zugenommen hat.

Schauen wir uns die Residuenplots an... Glauben Sie, dass die Punkte zufällig verteilt sind oder können Sie immer noch ein Muster erkennen?

In [ ]:
error_analysis(y_test, y_pred_test)

Wir sehen, dass die Residuen zwar etwas kleiner geworden sind (lineare Regression hatte Residuen von -100 bis 60), wir aber immer noch einige offensichtliche Muster erkennen können. 
Das Hinzufügen noch höherer Polynomgrade könnte helfen, dieses Muster zu erfassen...

### Lineares Modell mit Polynomen 3. Grades



Fügen wir mehr Flexibilität hinzu. Wir können auch einen weiteren Grad hinzufügen und eine kubische Kurve (degree=3) an den Datensatz anpassen. In der nächsten Zelle können Sie versuchen, es selbst zu tun.

In [ ]:
# here we will add the second degree polynomial features
polynomial_features = PolynomialFeatures(degree=3, include_bias=False)
x_poly_train = polynomial_features.fit_transform(X_train)
x_poly_test = polynomial_features.transform(X_test)

model_3_poly = LinearRegression()
model_3_poly.fit(x_poly_train, y_train)
y_pred_test = model_3_poly.predict(x_poly_test)
y_pred_train = model_3_poly.predict(x_poly_train)

In [ ]:
print("Polynomial Regression Model (degree 3):")
print("---"*10)
calculate_metrics(y_train,y_pred_train, y_test, y_pred_test)

In [ ]:
line_poly_3_reg = model_3_poly.intercept_ + model_3_poly.coef_[0][0] * x_plot + model_3_poly.coef_[0][1] * x_plot**2 + model_3_poly.coef_[0][2] * x_plot**3

In [ ]:
# Plotting 
plt.scatter(X_train, y_train, s=10,c="#193251",alpha=0.7, label="Train data")
plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",alpha=0.7, label= '"True" Relationship')
plt.plot(x_plot,line_lin_reg,c="#FFBA08",alpha=0.5,label= 'Linear model')
plt.plot(x_plot,line_poly_2_reg,c="#FFBA08", alpha=0.5,label= 'Degree 2 model')
plt.plot(x_plot,line_poly_3_reg,c="#FFBA08",marker = "o",ms=1, alpha=0.8,label= 'Degree 3 model')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Model Comparison")
plt.show();

In diesem Diagramm können wir sehen, dass das lineare Modell mit kubischen Merkmalen fast identisch mit dem Datengenerierungsprozess ist.
Der RMSE für die Trainings- und Testdaten ist auch im Vergleich zu den anderen Modellen niedrig.


| Metrik   | RMSE  | R²  | 
|---|---|---|
| **Lineares Modell**   | 44       |           0.79 | 
| **Polynomiales Modell (Grad 2)**     |34     |            0.87 | 
| **Polynomiales Modell (Grad 3)**     |19     |            0.96 |

In [ ]:
error_analysis(y_test, y_pred_test)

Wenn wir die Residuenplots überprüfen, sehen wir, dass der Bereich der Residuen viel kleiner ist und kein allgemeines Muster sichtbar ist. Dies deutet darauf hin, dass das Modell alle Beziehungen zwischen Merkmalen und Zielvariable erfasst, die in den Daten vorhanden sind.


### Lineares Modell mit Polynomen 8. Grades



Lassen Sie uns ein noch flexibleres Modell ausprobieren. Dieses Modell sollte auch in der Lage sein, die Muster in den Daten zu erfassen.

In [ ]:
# here we will add the second degree polynomial features
polynomial_features = PolynomialFeatures(degree=8, include_bias=False)
x_poly_train = polynomial_features.fit_transform(X_train)
x_poly_test = polynomial_features.transform(X_test)

model_8_poly = LinearRegression()
model_8_poly.fit(x_poly_train, y_train)
y_pred_test = model_8_poly.predict(x_poly_test)
y_pred_train = model_8_poly.predict(x_poly_train)

In [ ]:
print("Polynomial Regression Model (degree 8):")
print("---"*10)
calculate_metrics(y_train,y_pred_train, y_test, y_pred_test)

Was wir hier sehen, ist ein typisches Zeichen für Overfitting! Der RMSE auf den Trainingsdaten ist ziemlich niedrig: mit 14 sogar niedriger als beim vorherigen Modell (15). Aber auf der anderen Seite ist der RMSE auf den unbekannten Testdaten enorm gestiegen! 
Overfitting tritt auf, wenn das Modell unwissentlich einen Teil der Residualvariation (d.h. das Rauschen) extrahiert hat, als ob diese Variation die zugrunde liegende Beziehung darstellen würde. Die Verwendung dieses Modells wird keine guten Ergebnisse liefern, da die im Modell enthaltenen irrelevanten Informationen (Rauschen) nicht helfen werden, y für unbekannte Daten vorherzusagen.

| Metrik   | RMSE  | R²  | 
|---|---|---|
| **Lineares Modell**   | 44       |           0.79 | 
| **Polynomiales Modell (Grad 2)**     |34     |            0.87 | 
| **Polynomiales Modell (Grad 3)**     |19     |            0.96 | 
| **Polynomiales Modell (Grad 8)**     |33     |            0.88 | 


In [ ]:
line_poly_8_reg = model_8_poly.intercept_ + model_8_poly.coef_[0][0] * x_plot + model_8_poly.coef_[0][1] * x_plot**2 + model_8_poly.coef_[0][2] * x_plot**3 + model_8_poly.coef_[0][3] * x_plot**4 + model_8_poly.coef_[0][4] * x_plot**5 + model_8_poly.coef_[0][5] * x_plot**6 + model_8_poly.coef_[0][6] * x_plot**7 + model_8_poly.coef_[0][7] * x_plot**8

In [ ]:
# Plotting 
plt.scatter(X_train, y_train, s=10,c="#193251",alpha=0.7, label="Train data")
plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",alpha=0.7, label= '"True" Relationship')
#plt.plot(x_plot,line_lin_reg,c="#FFBA08",alpha=0.5,label= 'Linear model')
#plt.plot(x_plot,line_poly_2_reg,c="#FFBA08", alpha=0.5,label= 'Degree 2 model')
#plt.plot(x_plot,line_poly_3_reg,c="#FFBA08", alpha=0.5,label= 'Degree 3 model')
plt.plot(x_plot,line_poly_8_reg,c="#FFBA08",marker = "o",ms=1, alpha=0.8,label= 'Degree 8 model')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Model Comparison")
plt.show();

Dieses Modell ist viel flexibler als die anderen Modelle, die wir bisher verwendet haben, es schlängelt sich mehr, um die Trainingsdaten anzupassen. Das Modell mit Polynommerkmalen bis zu Grad 8 ist weiter von unserem Datengenerierungsprozess entfernt als das vorherige Modell.
Wir sehen, dass diese Linie die Trainingsdaten sehr gut passen könnte, aber sie passt nicht mehr so gut für unbekannte Testdaten.

Schauen wir uns nur die Trainingsdaten für unsere 4 Modelle an.

In [ ]:
# Plotting 
plt.scatter(X_train, y_train, s=10,c="#193251",alpha=0.7, label="Train data")
#plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",alpha=0.7, label= '"True" Relationship')
plt.plot(x_plot,line_lin_reg,c="#FFBA08",alpha=0.5,label= 'Linear model')
#plt.plot(x_plot,line_poly_2_reg,c="#FFBA08", alpha=0.5,label= 'Degree 2 model')
plt.plot(x_plot,line_poly_3_reg,c="#FFBA08", alpha=0.5,label= 'Degree 3 model')
plt.plot(x_plot,line_poly_8_reg,c="#FFBA08",marker = "o",ms=1, alpha=0.8,label= 'Degree 8 model')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Model Comparison")
plt.show();

Wir können sehen, dass die Modelle mit zunehmender Flexibilität die Trainingsdaten besser anpassen. Während das lineare Modell mit Polynomen der Ordnung 1 eindeutig versagt, die Variation in den Daten zu erfassen, passen die linearen Modelle mit Polynomen bis zur Ordnung 8 die Trainingsdaten sehr gut an, könnten aber auch Schwierigkeiten mit neuen Instanzen haben (= **Generalisierung**).



In [ ]:
error_analysis(y_test, y_pred_test)

Der Residuenplot für dieses Modell sieht für die meisten Teile nicht völlig schlecht aus. Aber es gab zwei Instanzen, bei denen unser Modell extrem schlecht vorhergesagt hat. 



## "High-Variance"-Modell


Ok... schauen wir uns noch einmal an, was "High-Variance"-Modelle bedeuten.

Varianz ist die Menge, um die sich die Schätzung des Ziels ändern wird, wenn andere Trainingsdaten verwendet würden. Wie empfindlich ist der Algorithmus auf die spezifischen verwendeten Trainingsdaten.

Um zu überprüfen, was dies in unserem Fall hier genau bedeutet, werden wir jetzt 3 nicht flexible Modelle (Polynomgrad von 1) und 3 hochflexible Modelle (Polynomgrad von 8) auf verschiedenen Trainingsdaten trainieren!
Und sehen, wie sehr sich diese Modelle voneinander unterscheiden.

Der erste Schritt besteht also darin, 2 weitere Train-Test-Splits zu erstellen, um 2 weitere Trainingssätze zu erhalten, mit denen wir unsere Modelle anpassen können.

In [ ]:
X_train_split2, X_test_split2, y_train_split2, y_test_split2 = train_test_split(x, y, test_size=0.8, random_state=91)
X_train_split3, X_test_split3, y_train_split3, y_test_split3 = train_test_split(x, y, test_size=0.8, random_state=222)

Jetzt werden wir zuerst diese neuen Daten verwenden, um 3 lineare Modelle zu trainieren:

In [ ]:
# Model 1
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Model 2
lin_reg_2 = LinearRegression()
lin_reg_2.fit(X_train_split2, y_train_split2)

# Model 3
lin_reg_3 = LinearRegression()
lin_reg_3.fit(X_train_split3, y_train_split3)


Lassen Sie uns diese drei Modelle plotten, um zu sehen, wie sehr sie sich voneinander unterscheiden!

In [ ]:
y_plot_true = 10 * x_plot - 2 * (x_plot ** 2) + 0.5 * (x_plot ** 3)
line_lin_reg = lin_reg.intercept_ + lin_reg.coef_[0]*x_plot
line_lin_reg_2 = lin_reg_2.intercept_ + lin_reg_2.coef_[0]*x_plot
line_lin_reg_3 = lin_reg_3.intercept_ + lin_reg_3.coef_[0]*x_plot

In [ ]:
# Plotting 
plt.scatter(x, y, s=10,c="#193251",alpha=0.7, label="All Data")
#plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",label= '"True" Relationship')
plt.plot(x_plot,line_lin_reg,c="#FFBA08",label= 'Linear model 1')
plt.plot(x_plot,line_lin_reg_2,c="#AAAE90",label= 'Linear model 2')
plt.plot(x_plot,line_lin_reg_3,c="#FFFA54",label= 'Linear model 3')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Comparison of linear models using different training data")
plt.show();

Die drei linearen Modelle sind nicht identisch. Aber die Achsenabschnitte und die Steigungen sind ziemlich ähnlich. 

Diese Modelle sind nicht sehr empfindlich gegenüber Änderungen in den Trainingsdaten.

Schauen wir uns an, wie die flexibleren Modelle abschneiden.

In [ ]:
polynomial_features = PolynomialFeatures(degree=8, include_bias=False)
x_poly_train = polynomial_features.fit_transform(X_train)
x_poly_test = polynomial_features.transform(X_test)

polynomial_features = PolynomialFeatures(degree=8, include_bias=False)
x_poly_train_2 = polynomial_features.fit_transform(X_train_split2)
x_poly_test_2 = polynomial_features.transform(X_test_split2)

polynomial_features = PolynomialFeatures(degree=8, include_bias=False)
x_poly_train_3 = polynomial_features.fit_transform(X_train_split3)
x_poly_test_3 = polynomial_features.transform(X_test_split3)


# Flexible model 1
poly_reg = LinearRegression()
poly_reg.fit(x_poly_train, y_train)

# Flexible model 2
poly_reg_2 = LinearRegression()
poly_reg_2.fit(x_poly_train_2, y_train_split2)

# Flexible model 3
poly_reg_3 = LinearRegression()
poly_reg_3.fit(x_poly_train_3, y_train_split3)

In [ ]:
y_plot_true = 10 * x_plot - 2 * (x_plot ** 2) + 0.5 * (x_plot ** 3)
line_poly_1 = poly_reg.intercept_ + poly_reg.coef_[0][0] * x_plot + poly_reg.coef_[0][1] * x_plot**2 + poly_reg.coef_[0][2] * x_plot**3 + poly_reg.coef_[0][3] * x_plot**4 + poly_reg.coef_[0][4] * x_plot**5 + poly_reg.coef_[0][5] * x_plot**6 + poly_reg.coef_[0][6] * x_plot**7 + poly_reg.coef_[0][7] * x_plot**8
line_poly_2 = poly_reg_2.intercept_ + poly_reg_2.coef_[0][0] * x_plot + poly_reg_2.coef_[0][1] * x_plot**2 + poly_reg_2.coef_[0][2] * x_plot**3 + poly_reg_2.coef_[0][3] * x_plot**4 + poly_reg_2.coef_[0][4] * x_plot**5 + poly_reg_2.coef_[0][5] * x_plot**6 + poly_reg_2.coef_[0][6] * x_plot**7 + poly_reg_2.coef_[0][7] * x_plot**8
line_poly_3 = poly_reg_3.intercept_ + poly_reg_3.coef_[0][0] * x_plot + poly_reg_3.coef_[0][1] * x_plot**2 + poly_reg_3.coef_[0][2] * x_plot**3 + poly_reg_3.coef_[0][3] * x_plot**4 + poly_reg_3.coef_[0][4] * x_plot**5 + poly_reg_3.coef_[0][5] * x_plot**6 + poly_reg_3.coef_[0][6] * x_plot**7 + poly_reg_3.coef_[0][7] * x_plot**8

In [ ]:
# Plotting 
plt.scatter(x, y, s=10,c="#193251",alpha=0.7, label="All data")
#plt.scatter(X_test, y_test, s=10,c="#FF5A36",alpha=0.7, label="Test data")
plt.plot(x_plot,y_plot_true,c="#5E5E5E",label= '"True" Relationship')
plt.plot(x_plot,line_poly_1,c="#FFBA08",label= 'Polynomial model 1')
plt.plot(x_plot,line_poly_2,c="b",label= 'Polynomial model 2')
plt.plot(x_plot,line_poly_3,c= "r",label= 'Polynomial model 3')
plt.legend()
plt.xlabel("feature x")
plt.ylabel("target variable y")
plt.title("Comparison of highly flexible models using different training data")
plt.show();

Wir können wieder sehen, dass diese flexiblen Modelle, obwohl sie die Trainingsdaten sehr gut anpassen, nicht gut generalisieren; sie werden nicht in der Lage sein, unbekannte Daten gut vorherzusagen. 
Diese flexiblen Modelle ändern sich stark in Abhängigkeit von unterschiedlichen Trainingsdaten: Es sind High-Variance-Modelle! 

Um zu verstehen, wie diese Kurven zustande kamen, können Sie die Trainingsdaten für die einzelnen Kurven getrennt von den Testdaten plotten!



## Zusammenfassung



- In diesem Notebook haben Sie den typischen Data-Science-Workflow gesehen, beginnend mit der Aufteilung der Daten in Trainings- und Testsatz, dem Ausprobieren verschiedener Modelle und der Bewertung ihrer Leistung anhand einer festgelegten Metrik und einer Fehleranalyse.


- Sie haben gelernt, flexiblere lineare Modelle zu erstellen, indem Sie Polynommerkmale verwenden.


- Sie wissen, dass die Bias-Varianz-Abwägung ein grundlegendes Konzept im maschinellen Lernen ist, das eine wichtige Rolle bei der Anwendung jedes maschinellen Lernalgorithmus spielen wird. 


- Sie wissen, dass Bias die vereinfachenden Annahmen sind, die das Modell trifft, um die Zielfunktion leichter zu approximieren.


- Varianz ist die Menge, um die sich die Schätzung der Zielfunktion bei unterschiedlichen Trainingsdaten ändern wird.


- Die Bias-Varianz-Abwägung beschreibt die Tatsache, dass es sinnvoll sein kann, einen verzerrten Schätzer zu verwenden, solange er die Varianz reduziert, vorausgesetzt, unser Ziel ist es, die Fehlerfunktion zu minimieren.




## Überprüfen Sie Ihr Verständnis



1. Berechnen Sie den RMSE (für Trainings- und Testdaten) der letzten 6 Modelle (3 lineare Regressionen und 3 Polynomregressionen (Ordnung 8)). Was erwarten Sie?

2. Trainieren Sie (8) Polynommodelle mit unterschiedlichen Graden (von 1-8 Polynomgrad) von Merkmalen. Plotten Sie den Trainings-RMSE und Test-RMSE der Modelle über die Modellflexibilitäten (Polynomgrad 1-8) entsprechend dem allerersten Plot (von Wikipedia) in diesem Notebook. Welchen Trend sehen Sie im Trainings-RMSE und Test-RMSE und was erklärt diese Trends?

3. Nach dem Ausprobieren dieser verschiedenen Modelle, welches Modell würden Sie wählen und warum? 

4. Was können Sie tun, um Overfitting zu verhindern?